# VSA-Based LISP Interpreter Demo

This notebook demonstrates a LISP interpreter built on Vector Symbolic Architecture (VSA). Every data structure — atoms, cons cells, lists — is represented as a hypervector, and every operation is VSA algebra.

## 1. Setup

Install `kongming-rs-hv` (if not already installed), then import `LispEnv` and create a fresh in-memory environment. Each session gets a random namespace to isolate user data.

In [ ]:
%pip install -q kongming-rs-hv

In [ ]:
import uuid
from kongming_rs.lisp import LispEnv

ns = f"user-{uuid.uuid4().hex[:8]}"
env = LispEnv(namespace=ns)
print(f"LispEnv ready, model: {env.model()}, namespace: {ns}")

## 2. Atoms and Lists

`parse_display` parses an S-expression into hypervectors and prints it back. `QUOTE` returns data unevaluated.

In [ ]:
# Parse and display a list (no evaluation)
env.parse_display("(A B C)")

In [ ]:
# QUOTE returns its argument unevaluated
env.eval("(QUOTE (A B C))")

## 3. Primitives: CAR, CDR, CONS

`CAR` returns the first element, `CDR` returns the rest, and `CONS` constructs a pair.

In [ ]:
# CAR: first element of a list
env.eval("(CAR (QUOTE (A B C)))")

In [ ]:
# CDR: rest of a list
env.eval("(CDR (QUOTE (A B C)))")

In [ ]:
# CONS: construct a pair
env.eval("(CONS (QUOTE A) (QUOTE B))")

## 4. Predicates: ATOM, EQ

`ATOM` tests whether its argument is an atom (not a list). `EQ` tests equality of two atoms.

In [ ]:
# ATOM on an atom -> T
env.eval("(ATOM (QUOTE A))")

In [ ]:
# ATOM on a list -> F
env.eval("(ATOM (QUOTE (A B)))")

In [ ]:
# EQ: equal atoms -> T
env.eval("(EQ (QUOTE A) (QUOTE A))")

In [ ]:
# EQ: different atoms -> F
env.eval("(EQ (QUOTE A) (QUOTE B))")

## 5. Conditional: COND

`COND` evaluates clauses in order and returns the value of the first whose test is true (`T`).

In [ ]:
# First clause is false, second (T) is the default branch
env.eval("(COND ((EQ (QUOTE A) (QUOTE B)) (QUOTE NO)) (T (QUOTE YES)))")

## 6. Lambda and DEFINE

`LAMBDA` creates anonymous functions. `DEFINE` binds a name to a value in the environment.

In [ ]:
# Inline lambda: take the CAR of the argument
env.eval("((LAMBDA (X) (CAR X)) (QUOTE (A B C)))")

In [ ]:
# DEFINE a reusable function SECOND
env.eval("(DEFINE SECOND (LAMBDA (L) (CAR (CDR L))))")

In [ ]:
# Call the defined function
env.eval("(SECOND (QUOTE (X Y Z)))")

## 7. Recursion with LABEL

`LABEL` enables recursive functions by giving a lambda a self-referencing name. Use `eval_full` for recursive expressions — it evaluates repeatedly until the result stabilizes.

In [ ]:
# DEFINE a recursive LAST function using LABEL
env.eval("(DEFINE LAST (LAMBDA (L) ((LABEL REC (LAMBDA (X) (COND ((ATOM (CDR X)) (CAR X)) (T (REC (CDR X)))))) L)))")

In [ ]:
# eval_full drives recursive evaluation to completion
env.eval_full("(LAST (QUOTE (A B C)))")

## 8. Persistent Storage with Fjall

Pass a filesystem `path` to `LispEnv` to back the codebook with fjall, a persistent key-value store. Definitions survive across sessions.

In [ ]:
import tempfile

d = tempfile.mkdtemp()
print("Fjall storage path:", d)

fenv = LispEnv(path=d)

In [ ]:
# Works the same as the in-memory environment
fenv.eval("(CAR (QUOTE (X Y Z)))")

In [ ]:
fenv.eval("(DEFINE GREET (LAMBDA (X) (CONS (QUOTE HELLO) X)))")
fenv.eval("(GREET (QUOTE WORLD))")